## Prepare questions from NACC for testing GRPO model

Created by: Sahana Kowshik

Date: 05/09/2025

In [1]:
import pandas as pd
import json
import random
import numpy as np

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"

In [3]:
test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")

/scratch/5552643.1.cds-gpu-long/ipykernel_2461212/2975898205.py:1: DtypeWarning: Columns (20,22,24,28,43,45,50,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,155,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,398,400,420,422,431,444,453,571,599,607,663,679,696,699,716,727,733,808,819,825,892,948,949,957,958,959,960,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summa

In [4]:
def add_etpr_labels(row):
    value = row['NACCETPR']
    
    # AD
    if value == 1:
        row['ETPR'] = 'AD'

    # LBD
    elif value == 2:
        row['ETPR'] = 'LBD'

    # VD
    elif value == 8:
        row['ETPR'] = 'VD'

    # Prion disease (CJD, other)
    elif value == 12:
        row['ETPR'] = 'PRD'

    # FTLD and its variants, including CBD and PSP, with or without ALS
    elif value in [4, 5, 6, 7]:
        row['ETPR'] = 'FTLD'

    # NPH
    elif value == 14:
        row['ETPR'] = 'NPH'
    
    # SEF. seizure, epilepsy, etc.
    elif value in [17, 18, 23, 26, 27, 28, 29]:
        row['ETPR'] = 'SEF'

    # Psychiatric conditions
    elif value in [19, 20, 21, 22, 24, 25]:
        row['ETPR'] = 'PSY'

    # TBI
    elif value == 13:
        row['ETPR'] = 'TBI'

    # Other degenerative
    elif value not in [88, 99]:
        row['ETPR'] = 'ODE'

    else:
        row['ETPR'] = np.NaN

    return row


test = test.apply(add_etpr_labels, axis=1)

In [5]:
test['ID'] = test['NACCID']
test.drop(['NACCID'], axis=1, inplace=True)

/scratch/5552643.1.cds-gpu-long/ipykernel_2461212/3636380100.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['ID'] = test['NACCID']


In [6]:
columns = ['ID', 'patient_summary', 'diag_summary', 'visit_summary',  'question', 'options', 'ground_truth']

In [7]:
ids_used = set()

## Neuropathology

In [8]:
test_np = test[((test['NACCLEWY'].isin([0,1,2,3,4])) & (test['NPFTDTAU'].isin([0,1])) & (test['NPADNC'].isin([0,1,2,3])))].sample(frac=1, random_state=0).reset_index(drop=True)

In [9]:
# Neuropathology Question Variants
np_question_variants = [
    "Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological causes of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathologies responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathologies causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathologies underlying the patient's cognitive symptoms from the available options."
]

# Answer options for the Neuropathology question
np_options = """A. Alzheimer's disease neuropathology (AD)
B. Lewy body pathology (LBD)
C. Frontotemporal Lobar Degeneration with tau pathology (FTLD-tau) or other tauopathy (FTLD)
D. AD and LBD
E. AD and FTLD
F. LBD and FTLD
G. AD and LBD and FTLD
H. None of the above"""


In [10]:
# Function to add NP ground truth columns
def add_ground_truth(row):
    # Alzheimer's Disease
    if row['NPADNC'] in [2, 3]:
        row['NP_AD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NPADNC'] in [0, 1]:
        row['NP_AD'] = 0
        row['NP_AVAIL'] = 1

    # Lewy Body Disease
    if row['NACCLEWY'] in [1, 2, 3, 4]:
        row['NP_LBD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NACCLEWY'] == 0:
        row['NP_LBD'] = 0
        row['NP_AVAIL'] = 1

    # Frontotemporal Degeneration (FTLD-tau or other)
    if row['NPFTDTAU'] == 1:
        row['NP_FTD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NPFTDTAU'] == 0:
        row['NP_FTD'] = 0
        row['NP_AVAIL'] = 1

    return row


In [11]:
test_np['NPADNC'].value_counts()

NPADNC
3.0    942
2.0    409
1.0    374
0.0    277
Name: count, dtype: int64

In [12]:
# Apply the function across the training set
test_np = test_np.apply(add_ground_truth, axis=1)

# Check distribution of available neuropathology labels
test_np['NP_AVAIL'].value_counts()

NP_AVAIL
1    2002
Name: count, dtype: int64

In [13]:
len(test_np)

2002

In [14]:
# Function to assign question, options, and ground truth label
def add_np_question(row):
    # Define ground truth answer
    if (row['NP_AD'] == 1) and (row['NP_LBD'] == 1) and (row['NP_FTD'] == 1):
        row['ground_truth'] = "G"
    elif (row['NP_LBD'] == 1) and (row['NP_FTD'] == 1):
        row['ground_truth'] = "F"
    elif (row['NP_AD'] == 1) and (row['NP_FTD'] == 1):
        row['ground_truth'] = "E"
    elif (row['NP_AD'] == 1) and (row['NP_LBD'] == 1):
        row['ground_truth'] = "D"
    elif row['NP_FTD'] == 1:
        row['ground_truth'] = "C"
    elif row['NP_LBD'] == 1:
        row['ground_truth'] = "B"
    elif row['NP_AD'] == 1:
        row['ground_truth'] = "A"
    else:
        row['ground_truth'] = "H"

    # Randomly assign one of the question variants
    row['question'] = random.choice(np_question_variants)
    row['options'] = np_options

    return row

In [15]:
# Apply question and label assignment
test_np = test_np.apply(add_np_question, axis=1)

# Check ground truth distribution
test_np['ground_truth'].value_counts()

ground_truth
A    675
D    538
H    317
C    172
B    115
E     70
G     68
F     47
Name: count, dtype: int64

In [16]:
# Update set of NACCIDs used to avoid duplication later
np_ids = set(test_np['ID'])

# Optional: Check distribution of diagnosis codes
test_np['NACCUDSD'].value_counts()

NACCUDSD
4    1535
1     270
3     197
Name: count, dtype: int64

In [17]:
# Select final columns to keep
test_np = test_np[columns]  # Make sure 'columns' is defined somewhere earlier
test_np['Q_TYPE'] = 'Neuropath'

In [18]:
test_np.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_np.csv", index=False)

## Amylpet

In [44]:
# Filter training datA. exclude patients already used in NP task and keep only those with valid amyloid PET results
test_pet = (
    test
    .loc[
        (test['AMYLPET'].isin([0, 1]))
    ]
    .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)

# Check the distribution of amyloid PET labels
test_pet['AMYLPET'].value_counts()

AMYLPET
1.0    102
0.0     52
Name: count, dtype: int64

In [45]:
# Question variants for amyloid PET classification
pet_question_variants = [
    "Identify if the patient is amyloid positive based on the provided information.",
    "Determine whether this patient is amyloid positive using the given data.",
    "Analyze the patient information to identify amyloid positivity.",
    "Based on the details provided, identify if the patient shows amyloid positivity.",
    "Using the provided patient data, identify whether the patient is amyloid positive."
]

# Multiple-choice answer options
pet_options = """A. Yes
B. No"""

In [46]:
# Keys related to amyloid biomarkers that should be removed from the patient summary
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [47]:
# Function to add question, options, ground truth, and modify patient summary
def add_pet_question(row):
    # Set ground truth answer
    if row['AMYLPET'] == 1:
        row['ground_truth'] = "A"  # Amyloid positive
    elif row['AMYLPET'] == 0:
        row['ground_truth'] = "B"  # Amyloid negative

    # Randomly assign one of the question variants
    row['question'] = random.choice(pet_question_variants)

    # Remove specific amyloid-related keys from the patient summary
    json_file = json.loads(row['patient_summary'])
    for key in pet_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
            
        if key in json_file.get('CSF evidence', {}):
            json_file['CSF evidence'].pop(key)
    
    # Update patient summary
    row['patient_summary'] = json.dumps(json_file, indent=4)
    
    # Set options
    row["options"] = pet_options

    return row

In [48]:
# Apply the function across the PET training dataset
test_pet = test_pet.apply(add_pet_question, axis=1)

# Quick check: How many times each question variant was picked
print(test_pet['question'].value_counts())

# Check distribution of amyloid PET labels
print(test_pet['AMYLPET'].value_counts())

question
Based on the details provided, identify if the patient shows amyloid positivity.      34
Determine whether this patient is amyloid positive using the given data.              34
Using the provided patient data, identify whether the patient is amyloid positive.    32
Analyze the patient information to identify amyloid positivity.                       30
Identify if the patient is amyloid positive based on the provided information.        24
Name: count, dtype: int64
AMYLPET
1.0    102
0.0     52
Name: count, dtype: int64


In [49]:
print(test_pet['ground_truth'].value_counts())

ground_truth
A    102
B     52
Name: count, dtype: int64


In [50]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
pet_ids = set(test_pet['ID'])

# Optional: Check cognitive diagnosis distribution
test_pet['NACCUDSD'].value_counts()

NACCUDSD
4    137
1     10
3      7
Name: count, dtype: int64

In [51]:
# Select only necessary columns
test_pet = test_pet[columns]
test_pet['Q_TYPE'] = 'Amyloid PET'

In [52]:
# print(test[test['NACCID'] == 'NACC867075'].iloc[0]['patient_summary'])

In [53]:
for key in pet_keys_remove:
    for i, row in test_pet.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [54]:
test_pet.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_questions/test_pet.csv", index=False)

## Datscan

In [55]:
# test_dat = test[(~test['NACCID'].isin(np_ids.union(pet_ids))) & (test['DATSCAN'].isin([0,1]))].apply(lambda x: x.sample(n=200, random_state=42)).reset_index(drop=True)

In [56]:
# Filter training datA. exclude subjects already used for NP and PET tasks,
# and keep only those with valid DATScan results
test_dat = (
    test
    .loc[
        (test['DATSCAN'].isin([0, 1]))
    ]
    .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)

# Check the distribution of DATScan labels
test_dat['DATSCAN'].value_counts()


DATSCAN
0.0    21
1.0    12
Name: count, dtype: int64

In [57]:
# Question variants for DATScan classification
dat_question_variants = [
    "Classify if the patient is DatScan positive based on the provided information.",
    "Using the given data, determine whether the patient is DatScan positive.",
    "Analyze the provided information to classify the patient's DatScan status.",
    "Based on the information given, classify whether the patient is DatScan positive.",
    "From the provided patient details, identify if the patient is DatScan positive."
]

# Multiple-choice answer options
dat_options = """A. Yes
B. No"""

In [58]:
# Keys related to DATScan that should be removed from the patient summary
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]


In [59]:
# Function to add question, options, ground truth, and modify patient summary for DAT task
def add_dat_question(row):
    # Set ground truth answer
    if row['DATSCAN'] == 1:
        row['ground_truth'] = "A"  # DATScan positive
    elif row['DATSCAN'] == 0:
        row['ground_truth'] = "B"  # DATScan negative

    # Randomly assign one of the question variants
    row['question'] = random.choice(dat_question_variants)

    # Remove specific DATScan-related keys from the patient summary
    json_file = json.loads(row['patient_summary'])
    for key in dat_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
    
    # Update patient summary
    row['patient_summary'] = json.dumps(json_file, indent=4)
    
    # Set options
    row["options"] = dat_options

    return row


In [60]:
# Apply the function across the DAT training dataset
test_dat = test_dat.apply(add_dat_question, axis=1)

# Check how many times each question variant was selected
print(test_dat['question'].value_counts())

# Check distribution of DATScan labels
print(test_dat['NACCUDSD'].value_counts())

question
Classify if the patient is DatScan positive based on the provided information.       9
Analyze the provided information to classify the patient's DatScan status.           7
From the provided patient details, identify if the patient is DatScan positive.      6
Using the given data, determine whether the patient is DatScan positive.             6
Based on the information given, classify whether the patient is DatScan positive.    5
Name: count, dtype: int64
NACCUDSD
4    29
3     2
1     2
Name: count, dtype: int64


In [61]:
print(test_dat['ground_truth'].value_counts())

ground_truth
B    21
A    12
Name: count, dtype: int64


In [62]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
dat_ids = set(test_dat['ID'])

In [63]:
# Select only the necessary columns
test_dat = test_dat[columns]
test_dat['Q_TYPE'] = 'DATSCAN'

In [64]:
for key in dat_keys_remove:
    for i, row in test_dat.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [66]:
test_dat.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_questions/test_dat.csv", index=False)

## NACCTMCI

In [9]:
# Sample MCI patients, excluding those already selected for NP, PET, or DAT tasks
test_mci = (
    test
    .groupby('NACCTMCI', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 200), random_state=42))  # Safe sampling
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check the number of sampled patients per MCI subtype
test_mci['NACCTMCI'].value_counts()

/scratch/5124330.1.cds-gpu/ipykernel_359488/3640180369.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 200), random_state=42))  # Safe sampling


NACCTMCI
8    200
2    197
1    102
3     60
4     41
Name: count, dtype: int64

In [41]:
test_mci['NACCUDSD'].value_counts()

NACCUDSD
3    400
4    151
1     49
Name: count, dtype: int64

In [42]:
# Question variants for MCI subtype classification
mci_question_variants = [
    "Determine the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.",
    "Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype.",
    "Using the information given, classify the patient's mild cognitive impairment (MCI) subtype.",
    "From the patient details provided, determine which mild cognitive impairment (MCI) subtype applies.",
    "Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype."
]

# Answer options for MCI subtype
mci_options = """A. Amnestic MCI- single domain
B. Amnestic MCI- multiple domain
C. Non-amnestic MCI- single domain
D. Non-amnestic MCI- multiple domain
E. Not applicable (no diagnosis of MCI)"""

In [43]:
# Function to assign ground truth label and MCI question
def add_mci_question(row):
    if row['NACCTMCI'] == 1:
        row['ground_truth'] = "A"
    elif row['NACCTMCI'] == 2:
        row['ground_truth'] = "B"
    elif row['NACCTMCI'] == 3:
        row['ground_truth'] = "C"
    elif row['NACCTMCI'] == 4:
        row['ground_truth'] = "D"
    elif row['NACCTMCI'] == 8:
        row['ground_truth'] = "E"

    # Randomly assign one of the question variants
    row['question'] = random.choice(mci_question_variants)

    row['options'] = mci_options
    return row

In [44]:
# Apply the function to the MCI training set
test_mci = test_mci.apply(add_mci_question, axis=1)

In [45]:
# Quick checks
print(test_mci['ground_truth'].value_counts())
print(test_mci['question'].value_counts())
print(test_mci['NACCUDSD'].value_counts())

ground_truth
E    200
B    197
A    102
C     60
D     41
Name: count, dtype: int64
question
Determine the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.    134
From the patient details provided, determine which mild cognitive impairment (MCI) subtype applies.                     131
Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype.                          113
Using the information given, classify the patient's mild cognitive impairment (MCI) subtype.                            111
Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype.                      111
Name: count, dtype: int64
NACCUDSD
3    400
4    151
1     49
Name: count, dtype: int64


In [46]:
# Update set of used IDs to avoid reusing these patients elsewhere
mci_ids = set(test_mci['ID'])

In [47]:
# Select only the columns needed for modeling
test_mci = test_mci[columns]
test_mci['Q_TYPE'] = 'MCI subtype'

In [48]:
test_mci.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_mci.csv", index=False)

## NACCETPR

In [8]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
test_etpr = (
    test
    .loc[
        ~(test['ETPR'].isin(['ODE']))  # Exclude 'other degenerative etiologies'
    ]
    # .groupby('ETPR', group_keys=False)
    # .apply(lambda x: x.sample(n=min(max(len(x) - 150, 0), 1000), random_state=42))  # Sample up to 1000
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
test_etpr['ETPR'].value_counts()

ETPR
AD      3075
FTLD     782
LBD      406
VD       160
PRD       72
SEF       34
PSY       13
TBI        8
NPH        2
Name: count, dtype: int64

In [11]:
# Question variants for primary etiology identification
etiology_question_variants = [
    "Identify the primary etiology of the patient's cognitive impairment using the information provided.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment.",
    "From the given patient information, select the primary underlying etiology for the cognitive symptoms.",
    "Using the data provided, identify the dominant cause of the patient's cognitive decline.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information."
]

# Answer options for primary etiology prediction
etiology_options = """A. Alzheimer's disease (AD)
B. Lewy body disease (LBD)
C. Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTD)
D. Vascular brain injury or vascular dementia including stroke (VD)
E. Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)
F. Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)
G. Traumatic brain injury (TBI)
H. Normal-pressure hydrocephalus (NPH)
I. Prion disease (CJD, other) 
J. Not applicable (no cognitive impairment)"""

In [12]:
# Function to assign ground truth label and MCI question
def add_etpr_question(row):
    if row['ETPR'] == 'AD':
        row['ground_truth'] = "A"
    elif row['ETPR'] == 'LBD':
        row['ground_truth'] = "B"
    elif row['ETPR'] == 'FTLD':
        row['ground_truth'] = "C"
    elif row['ETPR'] == 'VD':
        row['ground_truth'] = "D"
    elif row['ETPR'] == 'SEF':
        row['ground_truth'] = "E"
    elif row['ETPR'] == 'PSY':
        row['ground_truth'] = "F"
    elif row['ETPR'] == 'TBI':
        row['ground_truth'] = "G"
    elif row['ETPR'] == 'NPH':
        row['ground_truth'] = "H"
    elif row['ETPR'] == 'PRD':
        row['ground_truth'] = "I"
    else:
        row['ground_truth'] = "J"

    # Randomly assign one of the question variants
    row['question'] = random.choice(etiology_question_variants)

    row['options'] = etiology_options
    return row

In [13]:
# Apply the function to the MCI training set
test_etpr = test_etpr.apply(add_etpr_question, axis=1)

In [14]:
print(test_etpr['ground_truth'].value_counts())
print(test_etpr['question'].value_counts())
print(test_etpr['NACCUDSD'].value_counts())

ground_truth
A    3075
C     782
J     773
B     406
D     160
I      72
E      34
F      13
G       8
H       2
Name: count, dtype: int64
question
From the given patient information, select the primary underlying etiology for the cognitive symptoms.             1107
Determine the primary etiology underlying the patient's cognitive impairment based on the provided information.    1103
Using the data provided, identify the dominant cause of the patient's cognitive decline.                           1062
Identify the primary etiology of the patient's cognitive impairment using the information provided.                1034
Based on the clinical details, determine the main cause of the patient's cognitive impairment.                     1019
Name: count, dtype: int64
NACCUDSD
4    4199
1     656
3     470
Name: count, dtype: int64


In [24]:
# Update set of used IDs to avoid reusing these patients elsewhere
etpr_ids = set(test_etpr['ID'])

In [25]:
# Select only the columns needed for modeling
test_etpr = test_etpr[columns]
test_etpr['Q_TYPE'] = 'Primary etiology'

In [26]:
test_etpr.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_etpr.csv", index=False)

## NACCUDSD

In [27]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
test_cog = (
    test
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
test_cog['NACCUDSD'].value_counts()

NACCUDSD
4    4243
1     656
3     498
Name: count, dtype: int64

In [28]:
# Question variants for primary etiology identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Answer options for primary etiology prediction
cog_options = """A. Normal Cognition (NC)
B. Mild Cognitive Impairment (MCI)
C. Dementia (DE)"""

In [29]:
# Function to assign ground truth label and MCI question
def add_cog_question(row):
    if row['NACCUDSD'] == 1:
        row['ground_truth'] = "A"
    if row['NACCUDSD'] == 3:
        row['ground_truth'] = "B"
    if row['NACCUDSD'] == 4:
        row['ground_truth'] = "C"
        
    # Randomly assign one of the question variants
    row['question'] = random.choice(cog_question_variants)

    row['options'] = cog_options
    return row

In [30]:
# Apply the function to the MCI training set
test_cog = test_cog.apply(add_cog_question, axis=1)

In [31]:
print(test_cog['ground_truth'].value_counts())
print(test_cog['question'].value_counts())

ground_truth
C    4243
A     656
B     498
Name: count, dtype: int64
question
Analyze the patient's information and identify their cognitive status.                     1119
From the information available, determine the correct cognitive status for the patient.    1108
Using the information provided, determine the patient's cognitive status.                  1103
Based on the provided clinical details, classify the patient's cognitive status.           1056
Identify the patient's cognitive status based on the given information.                    1011
Name: count, dtype: int64


In [32]:
# Update set of used IDs to avoid reusing these patients elsewhere
cog_ids = set(test_cog['ID'])

In [33]:
# Select only the columns needed for modeling
test_cog = test_cog[columns]
test_cog['Q_TYPE'] = 'Cognitive status'

In [34]:
test_cog.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_cog.csv", index=False)